In [1]:
%load_ext autoreload
%autoreload 2

## Gamma n K study:

In [3]:
from pycocotools.coco import COCO
from pathlib import Path
# from pycocotools.cocoeval import COCOeval # original implementation
from SIoU import COCOevalSIoU as COCOeval # modified implementation
import numpy as np

def evaluate_coco_results(annotations_path, results_path, iou_threshold=None):
    
    # Load COCO ground truth and detections
    cocoGt = COCO(annotations_path)
    cocoDt = cocoGt.loadRes(results_path)
    
    # Initialize COCOeval object
    cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')  # or 'segm' for segmentation
    
    # Set the IoU threshold
    if iou_threshold is not None:
        cocoEval.params.iouThrs = [iou_threshold]
    else:
        cocoEval.params.iouThrs = [x for x in np.linspace(0.3, 0.5, 20)]
    
    # Run evaluation
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()
    
    precision = cocoEval.stats[0].item()
    recall = cocoEval.stats[-4].item()
    f1_score = 2 * (precision * recall) / (precision + recall)
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score}
    

def get_precision_b5(results_path):
    EVALS = []
    
    for band in [5]:
        folder = Path(f"/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band}")
        results_paths = [p for p in folder.glob("**/*/*test_results.bbox.json")]
        results_paths = [x for x in results_paths if 'LR_0.0008' in x.as_posix()]
        results_paths = [x for x in results_paths if 'BS_3' in x.as_posix()]
        results_paths = [x for x in results_paths if 'ME_30' in x.as_posix()]

        for results_path in results_paths:
            annotations_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{band}.json'
            try:
                evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
                EVALS.append(evaluation_result)
            except Exception as e:
                evaluation_result = {}
            
            # write result in the folder of result_path 
            with open(results_path.parent / "SIoU_50_evaluation.json", "w") as f:
                f.write(str(evaluation_result))
            
            
    return np.mean([x['precision'] for x in EVALS])


def get_precision_b6(results_path):
    EVALS = []
    
    for band in [6]:
        folder = Path(f"/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band}")
        results_paths = [p for p in folder.glob("**/*/*test_results.bbox.json")]
        results_paths = [x for x in results_paths if 'LR_0.0009' in x.as_posix()]
        results_paths = [x for x in results_paths if 'BS_2' in x.as_posix()]
        results_paths = [x for x in results_paths if 'ME_30' in x.as_posix()]

        for results_path in results_paths:
            annotations_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{band}.json'
            try:
                evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
                EVALS.append(evaluation_result)
            except Exception as e:
                evaluation_result = {}
            
            # write result in the folder of result_path 
            with open(results_path.parent / "SIoU_50_evaluation.json", "w") as f:
                f.write(str(evaluation_result))
            
            
    return np.mean([x['precision'] for x in EVALS])

In [98]:
p1, p2 = get_precision_b5(None), get_precision_b6(None)

print("Precision b5", p1)
print("Precision b6", p2)
# difference:
print("Difference", p1 - p2)


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.26s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.50 | area=   all | maxDets=100 ] = 0.812
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.50 | area= small | maxDets=100 ] = 0.812
 Average Precision  (AP) @[ IoU=0.50:0.50 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.50 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.50 | area=   all | maxDets=  1 ] = 0.067
 Average Recall     (AR) @[ IoU=0.50:0.50 | area=   all | maxDets= 10 ] = 0.583
 Average Recall     (AR) @[ IoU=0.50:0.50 | area=   all | max

# ALL BANDS RUNNERS

IoU is set to 0.4

Gamma and K are set to -6 and 2 respectively.

In [4]:
iou_threshold = 0.4

for band in [x for x in range(1,13,1)]:
    folder = Path(f"/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b{band}")
    results_paths = [p for p in folder.glob("**/*/*test_results.bbox.json")]
    # results_paths = [x for x in results_paths if 'LR_0.0008' in x.as_posix()]
    # results_paths = [x for x in results_paths if 'BS_3' in x.as_posix()]
    # results_paths = [x for x in results_paths if 'ME_30' in x.as_posix()]

    for results_path in results_paths:
        annotations_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{band}.json'
        try:
            evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
        except Exception as e:
            evaluation_result = {}
        
        # write result in the folder of result_path 
        with open(results_path.parent / "SIoU_40_evaluation.json", "w") as f:
            f.write(str(evaluation_result))

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.51s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | max

In [6]:
folder = Path(f"/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Special")
results_paths = [p for p in folder.glob("**/*/*test_results.bbox.json")]

for results_path in results_paths:
    annotations_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_5.json'
    try:
        evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
    except Exception as e:
        evaluation_result = {}
    
    # write result in the folder of result_path 
    with open(results_path.parent / "SIoU_40_evaluation.json", "w") as f:
        f.write(str(evaluation_result))

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.27s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=   all | maxDets=100 ] = 0.812
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= small | maxDets=100 ] = 0.812
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets=  1 ] = 0.067
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets= 10 ] = 0.580
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | max

# Sentinel

In [2]:
from pycocotools.coco import COCO
from pathlib import Path
# from pycocotools.cocoeval import COCOeval # original implementation
from SIoU import COCOevalSIoU as COCOeval # modified implementation
import numpy as np

def evaluate_coco_results(annotations_path, results_path, iou_threshold=None):
    
    # Load COCO ground truth and detections
    cocoGt = COCO(annotations_path)
    cocoDt = cocoGt.loadRes(results_path)
    
    # Initialize COCOeval object
    cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')  # or 'segm' for segmentation
    
    # Set the IoU threshold
    if iou_threshold is not None:
        cocoEval.params.iouThrs = [iou_threshold]
    else:
        cocoEval.params.iouThrs = [x for x in np.linspace(0.3, 0.5, 20)]
    
    # Run evaluation
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()
    
    precision = cocoEval.stats[0].item()
    recall = cocoEval.stats[-4].item()
    f1_score = 2 * (precision * recall) / (precision + recall)
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score}

In [10]:
import os

os.listdir('/Data_large/marine/Datasets/VDS2Raw/annotations/')

['val__band_4.json',
 'train.json',
 'test__band_4.json',
 'val__band_2.json',
 'val__band_8.json',
 'train__band_4.json',
 'train__band_8.json',
 'test__band_3.json',
 'val__band_3.json',
 'test__band_8.json',
 'src',
 'test.json',
 'test__band_2.json',
 'combined_coco.json',
 'train__band_3.json',
 'train__band_2.json',
 'val.json']

In [18]:
iou_threshold = 0.4


band_num = 8
annotations_path = f'/Data_large/marine/Datasets/VDS2Raw/annotations/test__band_{band_num}.json'
results = [x for x in Path(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Single/perfect_b{band_num}').glob("**/*/*test_results.bbox.json")]

for results_path in results:

    try:
        evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
    except Exception as e:
        print(e)
        evaluation_result = {}
    
    # write result in the folder of result_path 
    with open(results_path.parent / "SIoU_40_evaluation.json", "w") as f:
        f.write(str(evaluation_result))


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=   all | maxDets=100 ] = 0.453
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= small | maxDets=100 ] = 0.445
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=medium | maxDets=100 ] = 0.651
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets=  1 ] = 0.247
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets= 10 ] = 0.634
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxD

In [24]:
iou_threshold = 0.4


band_num = 2
annotations_path = f'/Data_large/marine/Datasets/VDS2Raw/annotations/test__band_{band_num}.json'
results = [x for x in Path(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/Sentinel/Multi/perfect_b2_b4_b8').glob("**/*/*test_results.bbox.json")]

for results_path in results:

    try:
        evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
    except Exception as e:
        print(e)
        evaluation_result = {}
    
    # write result in the folder of result_path 
    with open(results_path.parent / "SIoU_40_evaluation.json", "w") as f:
        f.write(str(evaluation_result))


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=   all | maxDets=100 ] = 0.627
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= small | maxDets=100 ] = 0.604
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=medium | maxDets=100 ] = 0.832
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets=  1 ] = 0.290
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets= 10 ] = 0.677
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxD

# Venus Multi

In [32]:
iou_threshold = 0.4


band_num = 5
annotations_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{band_num}.json'

results = [x for x in Path(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Special/BS_2/LR_0.0015/IMG_2048/BANDS__b5_b10_b11').glob("**/*/*test_results.bbox.json")]

for results_path in results:

    try:
        evaluation_result = evaluate_coco_results(annotations_path, results_path.as_posix(), iou_threshold)
    except Exception as e:
        print(e)
        evaluation_result = {}
    
    # write result in the folder of result_path 
    with open(results_path.parent / "SIoU_40_evaluation.json", "w") as f:
        f.write(str(evaluation_result))


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.15s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=   all | maxDets=100 ] = 0.845
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= small | maxDets=100 ] = 0.845
 Average Precision  (AP) @[ IoU=0.40:0.40 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.40:0.40 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets=  1 ] = 0.067
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | maxDets= 10 ] = 0.596
 Average Recall     (AR) @[ IoU=0.40:0.40 | area=   all | max